# Missing Data Analysis: Contraceptive Method Choice

**Dataset**: Contraceptive Method Choice (UCI ML Repository #30)  
**Source**: Based on the 1987 National Indonesia Contraceptive Prevalence Survey.  
**Size**: 1,473 married women × 10 features

This notebook demonstrates how `missingfcup` visualisations can help identify the underlying
mechanism of missing data — **MCAR**, **MAR**, or **MNAR** — by generating controlled synthetic
missingness on an otherwise complete dataset.

The dataset records demographic and socioeconomic characteristics of married Indonesian women
alongside their reported contraceptive use. Survey data on family planning is inherently
susceptible to **social desirability bias (MNAR)** — sensitive values are selectively
under-reported — and to **demographic conditioning (MAR)** — where age or education predicts
whether other variables are observed. With only 10 variables, missingness patterns are
particularly compact and easy to communicate visually.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from missingfcup import MissingData
from mdatagen.multivariate.mMCAR import mMCAR
from mdatagen.multivariate.mMAR  import mMAR
from mdatagen.multivariate.mMNAR import mMNAR

contraceptive_method_choice = fetch_ucirepo(id=30)
X_raw = contraceptive_method_choice.data.features
y = contraceptive_method_choice.data.targets.iloc[:, 0].to_numpy()

# All features are already numeric — ordinal and binary variables are pre-encoded.
X = X_raw.copy().astype(float)
df = pd.concat([X, pd.Series(y, name="contraceptive_method")], axis=1)

print(f"Shape : {df.shape}")
print(f"Missing cells : {df.isna().sum().sum()}")
df.head()

## Complete Dataset — Baseline

Before injecting any missingness, we run all available `missingfcup` plots on the complete
original dataset. With only 10 variables, the plots are compact and easy to read — making
this dataset especially useful for illustrating the contrast between mechanisms. Plots that
require at least one missing value are skipped.

In [ ]:
md_complete = MissingData(df)

md_complete.overall_missingness_barchart().show()
md_complete.missing_count_barchart().show()
md_complete.column_missing_rate_heatmap().show()
md_complete.heatmap().show()
md_complete.parallel_coordinates().show()
md_complete.scatterplot(x="wife_age", y="num_children").show()

## Generating Synthetic Missingness

Since the dataset is complete, we inject missingness using **`mdatagen`**, generating three
copies with **30% missingness** under each of the three canonical mechanisms.

| Mechanism | Rule | Identifiability |
|-----------|------|-----------------|
| **MCAR** | Removed uniformly at random — independent of all values | Testable (Little's test) |
| **MAR**  | Missingness in some columns is driven by *observed* values of `wife_age` | Not directly testable; structural evidence needed |
| **MNAR** | Missingness depends on the *missing value itself* — low `standard_of_living` values are absent | Not identifiable from observed data alone |

The 10-variable structure of this dataset makes the contrast between mechanisms especially
legible: every heatmap and correlation matrix fits in a compact grid.

## 1. MCAR — Missing Completely At Random

Values are removed independently of all observed and unobserved data. Every cell has an equal
probability of being missing regardless of the woman's age, education, or family characteristics.

**Expected signatures:**
- Heatmap: uniform salt-and-pepper scatter — no row or column banding
- Missing-missing correlation: near-zero throughout
- Present-missing correlation: flat
- Parallel coordinates: missing values distributed uniformly across all value ranges
- Little's MCAR test: p > 0.05

In [ ]:
m_mcar = mMCAR(X=X, y=y, missing_rate=30)
data_m_mcar = m_mcar.random()
md_mcar = MissingData(data_m_mcar)

# --- Volume & distribution ---
md_mcar.overall_missingness_barchart().show()
md_mcar.missing_count_barchart().show()
md_mcar.column_missing_rate_heatmap().show()

# --- Structural pattern ---
md_mcar.heatmap().show()
md_mcar.barchart_intersection().show()
md_mcar.barchart_venn(selected_columns=["wife_age", "num_children", "wife_education"]).show()

# --- Correlation structure ---
md_mcar.missing_missing_heatmap().show()
md_mcar.present_missing_heatmap().show()
md_mcar.dendrogram().show()

# --- Value-level patterns ---
md_mcar.parallel_coordinates().show()
md_mcar.scatterplot(x="wife_age", y="num_children").show()

### What the Plots Reveal — MCAR

**Volume & Distribution**  
~30% of cells are missing, spread uniformly across all 10 columns. The missing count barchart
shows a flat profile — no column is systematically emptier than others. With only 10 variables,
this uniformity is immediately apparent at a glance.

**Structural Pattern**  
The heatmap shows a random scatter with no visible banding. The barchart intersection
confirms that most incomplete rows are missing in just one or two independently selected
columns — no dominant joint-missingness pattern emerges.

**Correlation Structure**  
All pairwise missing-missing correlations are near zero. With only 10 variables, the 10×10
correlation heatmap fits in a compact grid and the flatness is easy to read at a glance.
The present-missing heatmap is equally uninformative: no column pair shows a cross-signal.
The dendrogram is shallow with arbitrary leaf ordering.

**Value-Level Patterns**  
Missing `wife_age` and `num_children` values span the full observed ranges in the parallel
coordinates — the offset points appear across all ages and parity levels. The scatterplot
shows no clustering of missing points in any region of the `wife_age` × `num_children` space.

> **Mechanism Signature**: The compact 10-variable structure makes MCAR's flatness
> unmistakable. This dataset is ideal for side-by-side comparison: the transition from
> this structureless baseline to the MAR pattern below is immediately striking.

## 2. MAR — Missing At Random

Missingness depends on other *observed* variables but not on the value that is missing.
Here, missingness in education and contraceptive-related columns is conditioned on
`wife_age`: older women (age > 35) are more likely to have these values absent — simulating
a realistic survey scenario where older cohorts were less likely to complete certain
sections of the questionnaire.

**Expected signatures:**
- Heatmap: structured row banding — rows with similar `wife_age` cluster as jointly incomplete
- Missing-missing correlation: high positive correlations between conditioned columns
- Present-missing correlation: `wife_age` presence predicts missingness in education/contraceptive columns
- Parallel coordinates: missing values cluster in the high-`wife_age` band
- Little's MCAR test: p < 0.05

In [ ]:
m_mar = mMAR(X=X, y=y, n_xmiss=5)
data_m_mar = m_mar.random(missing_rate=30)
md_mar = MissingData(data_m_mar)

# --- Volume & distribution ---
md_mar.overall_missingness_barchart().show()
md_mar.missing_count_barchart().show()
md_mar.column_missing_rate_heatmap().show()

# --- Structural pattern ---
md_mar.heatmap().show()
md_mar.barchart_intersection().show()
md_mar.barchart_venn(selected_columns=["wife_age", "num_children", "wife_education"]).show()

# --- Correlation structure ---
md_mar.missing_missing_heatmap().show()
md_mar.present_missing_heatmap().show()
md_mar.dendrogram().show()

# --- Value-level patterns: colour by the MAR driver (wife_age) ---
md_mar.parallel_coordinates(missingness_color_column="wife_age").show()
md_mar.scatterplot(x="wife_age", y="num_children").show()

### What the Plots Reveal — MAR

**Volume & Distribution**  
Column-level missingness becomes unequal: conditioned columns (`wife_education`,
`contraceptive_method`) show elevated rates, while the driver column (`wife_age`) remains
fully observed. The missing count barchart makes this asymmetry immediately visible.

**Structural Pattern**  
The heatmap shows **clear horizontal banding**: women of similar ages cluster together as
jointly incomplete on conditioned variables. With only 10 features, this banding is
unambiguous and easy to point to. The barchart intersection shifts to a small number of
large, dominant patterns — a direct contrast with the dispersed MCAR profile.

**Correlation Structure**  
The 10×10 missing-missing correlation heatmap shows high positive correlations between
`wife_education`, `husband_education`, and `standard_of_living` — they miss together
because their missingness is conditioned on the same driver (`wife_age`). The dendrogram
produces tight, meaningful clusters.

The **present-missing heatmap** is the most diagnostic tool here: the `wife_age` row
shows a strong signal — its presence in a record predicts missingness in education and
contraceptive-related columns. In a 10×10 grid this driver column stands out at a glance.

**Value-Level Patterns**  
The parallel coordinates, coloured by `wife_age` missingness, shows offset points
clustering in the older-age range. The scatterplot of `wife_age` vs `num_children`
reveals missing `num_children` values concentrated among older women — the at-risk
demographic subgroup.

> **Mechanism Signature**: In a 10-variable dataset, MAR produces unmistakably clear
> banding and correlation structure. The present-missing heatmap is compact enough to
> pinpoint the driver column (`wife_age`) at a glance — making this the single most
> diagnostic plot for this mechanism.

## 3. MNAR — Missing Not At Random

Missingness depends on the *missing value itself*. Here, low values of `standard_of_living`
are more likely to be absent — simulating social desirability bias: women with low standards
of living under-report this sensitive variable. Similarly, extreme values of other socioeconomic
indicators may be systematically absent.

This mechanism is not identifiable from observed data alone: the information needed to explain
the missingness is precisely the information that is gone.

**Expected signatures:**
- Heatmap: superficially similar to MCAR — no demographic banding
- Missing-missing correlation: moderate but weaker than MAR
- Present-missing correlation: weak — no observed column cleanly predicts the missingness
- Parallel coordinates: missing values concentrate at the low end of the `standard_of_living` axis
- Little's MCAR test: p < 0.05 (cannot distinguish from MAR)

In [ ]:
m_mnar = mMNAR(X=X, y=y, threshold=1, n_xmiss=8)
data_m_mnar = m_mnar.random(missing_rate=30)
md_mnar = MissingData(data_m_mnar)

# --- Volume & distribution ---
md_mnar.overall_missingness_barchart().show()
md_mnar.missing_count_barchart().show()
md_mnar.column_missing_rate_heatmap().show()

# --- Structural pattern ---
md_mnar.heatmap().show()
md_mnar.barchart_intersection().show()
md_mnar.barchart_venn(selected_columns=["wife_age", "num_children", "wife_education"]).show()

# --- Correlation structure ---
md_mnar.missing_missing_heatmap().show()
md_mnar.present_missing_heatmap().show()
md_mnar.dendrogram().show()

# --- Value-level patterns: the clearest MNAR signal ---
md_mnar.parallel_coordinates().show()
md_mnar.scatterplot(x="wife_age", y="num_children").show()

### What the Plots Reveal — MNAR

**Volume & Distribution**  
Missing rates are elevated in socioeconomically sensitive columns, but the overall volume
is similar to the other scenarios. Columns with more extreme value distributions (e.g.,
`standard_of_living`) show higher missing rates than columns with more uniform distributions.

**Structural Pattern**  
The heatmap shows **no row banding** — missing values appear scattered without the
demographic clustering we saw under MAR. This is the central visual challenge: MNAR can
look like MCAR from the structural pattern alone. The contrast with the MAR heatmap is
stark and immediately visible in this compact dataset.

**Correlation Structure**  
The 10×10 missing-missing heatmap shows weaker, less structured correlations than MAR.
Columns affected by similar threshold rules may co-miss to some extent, but the signal
is noisier. The present-missing heatmap shows no clear driver column — because the true
predictor (the missing column's own value) is unobserved.

**Value-Level Patterns**  
The parallel coordinates reveal the MNAR signature: missing `standard_of_living` values
cluster at the **low end** of the visible range — women present in the low-SES region are
less likely to have this variable recorded. This is a realistic social desirability pattern.
The scatterplot hints at MNAR: missing `num_children` values tend to cluster toward higher
parity levels (women with many children are selectively absent).

> **Mechanism Signature**: In this compact dataset, the contrast between MAR banding and
> MNAR's unstructured scatter is especially clear. The parallel coordinates — by showing
> WHERE in the value range missing points accumulate — remains the best tool for detecting
> MNAR when domain knowledge guides which variables to examine.

## Comparative Analysis — Little's MCAR Test

Little's test evaluates whether the missingness pattern is consistent with MCAR.

| Result | Interpretation |
|--------|-----------------|
| **p > 0.05** | Fail to reject MCAR |
| **p ≤ 0.05** | Reject MCAR — consistent with MAR or MNAR |

**Important**: The test cannot distinguish MAR from MNAR. Visual tools are required to go further.

In [ ]:
results = {
    "MCAR": md_mcar.littles_mcar_test(),
    "MAR":  md_mar.littles_mcar_test(),
    "MNAR": md_mnar.littles_mcar_test(),
}

pd.DataFrame({k: v[["chi2", "df", "p_value"]] for k, v in results.items()}).T

## Conclusions

### Plot Utility by Mechanism

| Plot | MCAR | MAR | MNAR |
|------|------|-----|------|
| Binary heatmap | Baseline | ★★★ Clear demographic banding | ★ Unstructured |
| Missing-missing correlation | ★ Flat | ★★★ Tight clusters | ★★ Moderate, noisy |
| Present-missing heatmap | ★ Flat | ★★★ Driver column obvious in 10×10 grid | ★ Weak signal |
| Parallel coordinates | ★ Uniform | ★★ Age-banded | ★★★ Low-value clustering |
| Scatterplot | ★ Random | ★★ Age-concentrated | ★★ High-parity bias |
| Little's MCAR test | ★★★ Confirmed | ★★★ Rejected | ★★ Rejected — indistinguishable from MAR |

### Key Takeaways

1. **Small datasets amplify visual clarity.** With 10 variables, the transition from MCAR
   (no pattern) to MAR (clear banding) is immediately striking. The present-missing heatmap
   fits in a compact 10×10 grid that is easy to scan and explain to any audience.

2. **Survey data naturally produces MAR and MNAR.** The `wife_age` → education missingness
   (MAR) and `standard_of_living` → self-reporting bias (MNAR) reflect real-world mechanisms
   commonly encountered in demographic survey analysis.

3. **MNAR requires domain-guided investigation.** Without knowing that `standard_of_living`
   is a socially sensitive variable, a naive analyst might classify the MNAR case as MCAR
   based on correlation-only tools. The parallel coordinates plot, combined with domain
   knowledge, is the decisive instrument.

4. **The present-missing heatmap is the single most diagnostic plot for MAR.** In this
   compact dataset, the driver column (`wife_age`) stands out unambiguously in the 10×10
   grid — impossible to overlook.

5. **Little's test confirms but does not diagnose.** The combination of test + visual tools
   provides the most complete and reliable picture of which mechanism is operating.